# Revoxelized Open3D Mesh: Dense vs Sparse Radius Neighbors

This notebook verifies dense and sparse radius-neighbor search on a mesh loaded by Open3D.

For each run, it samples a fresh random rotation, rotates the normalized mesh vertices, re-voxelizes the rotated mesh with Open3D at `grid_size = 1024`, rotates the new voxel centers back to the original pose, and compares dense vs sparse `(qid, kid)` pairs with `radius = 2`.

The code is intentionally flat. There are no helper function definitions.

In [1]:
import os
import sys
import time
import gc
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "symtrellis").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import torch
import open3d as o3d

from symtrellis.geometry.neighbors import _sparse_lattice_ext
from symtrellis.geometry.coords import grid2pos
from symtrellis.geometry.neighbors.offsets import lattice_ball_offsets
from symtrellis.geometry.neighbors.dense_lattice import radius_nbr_edges_dense_lattice
from symtrellis.geometry.neighbors.sparse_lattice import radius_nbr_edges_sparse_lattice

print('ROOT =', ROOT)
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
print('sparse extension =', _sparse_lattice_ext.__file__)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.
ROOT = <repo-root>
torch = 2.6.0+cu124
cuda available = True
sparse extension = <repo-root>/symtrellis/geometry/neighbors/_sparse_lattice_ext.cpython-310-x86_64-linux-gnu.so


In [2]:
mesh_path = Path(os.environ["SYMTRELLIS_TEST_MESH"])
grid_size = 1024
radius = 2.0
n_runs = 5
seed = 12345
device = torch.device('cuda')

assert mesh_path.exists(), mesh_path
assert torch.cuda.is_available()

print('mesh_path =', mesh_path)
print('grid_size =', grid_size)
print('radius =', radius)
print('n_runs =', n_runs)
print('seed =', seed)

mesh_path = <SYMTRELLIS_TEST_MESH>
grid_size = 1024
radius = 2.0
n_runs = 5
seed = 12345


## Load and Normalize Mesh

The mesh is centered by its bounding-box center and divided by its longest bounding-box side, so the longest axis fits into `[-0.5, 0.5]`.

In [3]:
mesh = o3d.io.read_triangle_mesh(str(mesh_path), enable_post_processing=True)
vertices = np.asarray(mesh.vertices, dtype=np.float64)
triangles = np.asarray(mesh.triangles)

bbox_min = vertices.min(axis=0)
bbox_max = vertices.max(axis=0)
bbox_center = (bbox_min + bbox_max) * 0.5
bbox_scale = (bbox_max - bbox_min).max()
vertices_norm = (vertices - bbox_center) / bbox_scale

mesh.vertices = o3d.utility.Vector3dVector(vertices_norm)
mesh.compute_vertex_normals()
base_vertices = np.asarray(mesh.vertices, dtype=np.float64).copy()

print('vertices =', vertices.shape)
print('triangles =', triangles.shape)
print('original bbox min =', bbox_min)
print('original bbox max =', bbox_max)
print('normalized bbox min =', base_vertices.min(axis=0))
print('normalized bbox max =', base_vertices.max(axis=0))

vertices = (295724, 3)
triangles = (490086, 3)
original bbox min = [-0.50198948 -0.04447482 -0.47620469]
original bbox max = [0.50195312 0.04431351 0.47540894]
normalized bbox min = [-0.5        -0.04421982 -0.47393826]
normalized bbox max = [0.5        0.04421982 0.47393826]


## Open3D Voxelize Original Pose Once

The original-pose voxel grid is the key set. Open3D can emit a few indices outside the fixed grid near the boundary, so the code filters to `0..1023` and then deduplicates with `np.unique`.

In [4]:
voxel_size = 1.0 / grid_size
min_bound = np.array([-0.5, -0.5, -0.5], dtype=np.float64)
max_bound = np.array([0.5, 0.5, 0.5], dtype=np.float64)

t0 = time.time()
voxel_grid_key = o3d.geometry.VoxelGrid.create_from_triangle_mesh_within_bounds(
    mesh, voxel_size, min_bound, max_bound
)
key_np_raw = np.asarray(
    [voxel.grid_index for voxel in voxel_grid_key.get_voxels()], dtype=np.int32
).reshape(-1, 3)
valid_key = ((key_np_raw >= 0) & (key_np_raw < grid_size)).all(axis=1)
key_np = np.unique(key_np_raw[valid_key], axis=0)
key_voxelize_seconds = time.time() - t0

key_coords = torch.from_numpy(key_np).to(device=device, dtype=torch.int32)
key_bid = torch.zeros((key_coords.shape[0],), device=device, dtype=torch.int32)
nbr_offsets = lattice_ball_offsets(radius, device=device)
coord_min = torch.tensor([0, 0, 0], device=device, dtype=torch.int32)

print('key raw voxels =', key_np_raw.shape[0])
print('key valid unique voxels =', key_np.shape[0])
print('key dropped outside grid =', key_np_raw.shape[0] - int(valid_key.sum()))
print('key min =', key_np.min(axis=0))
print('key max =', key_np.max(axis=0))
print('key voxelize seconds =', key_voxelize_seconds)
print('L =', nbr_offsets.shape[0])

key raw voxels = 3628352
key valid unique voxels = 3628351
key dropped outside grid = 1
key min = [  0 466  26]
key max = [1023  557  997]
key voxelize seconds = 6.422421216964722
L = 88


## Repeated Random Rotations

Each run creates a fresh random rotation, rotates the normalized mesh vertices, re-voxelizes that rotated mesh with Open3D, rotates the resulting voxel centers back, and then times dense and sparse radius-neighbor search.

In [5]:
rng = np.random.default_rng(seed)
results = []

for run_id in range(n_runs):
    quat = rng.normal(size=4)
    quat = quat / np.linalg.norm(quat)
    w, x, y, z = quat

    R_np = np.array([
        [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
        [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
        [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
    ], dtype=np.float64)

    mesh_rot = o3d.geometry.TriangleMesh(mesh)
    mesh_rot.vertices = o3d.utility.Vector3dVector(base_vertices @ R_np.T)
    mesh_rot.compute_vertex_normals()

    t0 = time.time()
    voxel_grid_query = o3d.geometry.VoxelGrid.create_from_triangle_mesh_within_bounds(
        mesh_rot, voxel_size, min_bound, max_bound
    )
    query_grid_np_raw = np.asarray(
        [voxel.grid_index for voxel in voxel_grid_query.get_voxels()], dtype=np.int32
    ).reshape(-1, 3)
    valid_query = ((query_grid_np_raw >= 0) & (query_grid_np_raw < grid_size)).all(axis=1)
    query_grid_np = np.unique(query_grid_np_raw[valid_query], axis=0)
    voxelize_seconds = time.time() - t0

    query_grid = torch.from_numpy(query_grid_np).to(device=device, dtype=torch.int32)
    R = torch.from_numpy(R_np).to(device=device, dtype=torch.float64)
    query_pos_rot = grid2pos(query_grid, grid_size).to(torch.float64)
    query_pos_norm = query_pos_rot @ R
    query_pos = ((query_pos_norm + 0.5) * grid_size - 0.5).contiguous()
    query_bid = torch.zeros((query_pos.shape[0],), device=device, dtype=torch.int32)

    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats(device)
    dense_start_bytes = torch.cuda.memory_allocated(device)
    t0 = time.time()
    q_dense, k_dense = radius_nbr_edges_dense_lattice(
        query_pos=query_pos,
        query_bid=query_bid,
        key_coords=key_coords,
        key_bid=key_bid,
        radius=radius,
        nbr_offsets=nbr_offsets,
        grid_size=grid_size,
        chunk_size=8192,
    )
    torch.cuda.synchronize()
    dense_seconds = time.time() - t0
    dense_peak_bytes = torch.cuda.max_memory_allocated(device)
    dense_peak_mb = dense_peak_bytes / 1024 ** 2
    dense_extra_peak_mb = (dense_peak_bytes - dense_start_bytes) / 1024 ** 2
    dense_pair = torch.sort(q_dense * key_coords.shape[0] + k_dense).values.cpu()
    dense_edge_count = q_dense.numel()
    del q_dense
    del k_dense
    gc.collect()
    torch.cuda.empty_cache()

    torch.cuda.reset_peak_memory_stats(device)
    sparse_start_bytes = torch.cuda.memory_allocated(device)
    t0 = time.time()
    q_sparse, k_sparse = radius_nbr_edges_sparse_lattice(
        query_pos=query_pos,
        query_bid=query_bid,
        key_coords=key_coords,
        key_bid=key_bid,
        radius=radius,
        nbr_offsets=nbr_offsets,
        coord_min=coord_min,
    )
    torch.cuda.synchronize()
    sparse_seconds = time.time() - t0
    sparse_peak_bytes = torch.cuda.max_memory_allocated(device)
    sparse_peak_mb = sparse_peak_bytes / 1024 ** 2
    sparse_extra_peak_mb = (sparse_peak_bytes - sparse_start_bytes) / 1024 ** 2
    sparse_pair = torch.sort(q_sparse * key_coords.shape[0] + k_sparse).values.cpu()
    sparse_edge_count = q_sparse.numel()
    del q_sparse
    del k_sparse
    gc.collect()
    torch.cuda.empty_cache()

    same_count = dense_pair.numel() == sparse_pair.numel()
    same_pairs = torch.equal(dense_pair, sparse_pair)

    results.append({
        'run': run_id,
        'query_voxels': int(query_grid_np.shape[0]),
        'dropped_query_voxels': int(query_grid_np_raw.shape[0] - valid_query.sum()),
        'edges': int(dense_edge_count),
        'voxelize_seconds': float(voxelize_seconds),
        'dense_seconds': float(dense_seconds),
        'sparse_seconds': float(sparse_seconds),
        'dense_peak_mb': float(dense_peak_mb),
        'sparse_peak_mb': float(sparse_peak_mb),
        'dense_extra_peak_mb': float(dense_extra_peak_mb),
        'sparse_extra_peak_mb': float(sparse_extra_peak_mb),
        'same_count': bool(same_count),
        'same_pairs': bool(same_pairs),
        'det_R': float(np.linalg.det(R_np)),
    })

    print(
        'run', run_id,
        'query_voxels', query_grid_np.shape[0],
        'edges', dense_edge_count,
        'voxelize_s', round(voxelize_seconds, 4),
        'dense_s', round(dense_seconds, 4),
        'sparse_s', round(sparse_seconds, 4),
        'dense_extra_mb', round(dense_extra_peak_mb, 1),
        'sparse_extra_mb', round(sparse_extra_peak_mb, 1),
        'same_pairs', same_pairs,
    )

    assert same_count
    assert same_pairs

    del query_grid
    del query_pos_rot
    del query_pos_norm
    del query_pos
    del query_bid
    del dense_pair
    del sparse_pair
    gc.collect()
    torch.cuda.empty_cache()

run 0 query_voxels 4619456 edges 62848719 voxelize_s 8.5022 dense_s 0.2005 sparse_s 0.0421 dense_extra_mb 10182.6 sparse_extra_mb 6666.0 same_pairs True
run 1 query_voxels 4513573 edges 61052038 voxelize_s 8.6653 dense_s 0.1778 sparse_s 0.0386 dense_extra_mb 10133.6 sparse_extra_mb 6523.1 same_pairs True
run 2 query_voxels 3773820 edges 51990377 voxelize_s 7.7324 dense_s 0.1604 sparse_s 0.0356 dense_extra_mb 9841.8 sparse_extra_mb 5529.5 same_pairs True
run 3 query_voxels 4208605 edges 57806222 voxelize_s 7.6409 dense_s 0.1607 sparse_s 0.0352 dense_extra_mb 10026.6 sparse_extra_mb 6113.5 same_pairs True
run 4 query_voxels 4420296 edges 60903873 voxelize_s 8.3229 dense_s 0.1696 sparse_s 0.0395 dense_extra_mb 10112.3 sparse_extra_mb 6397.5 same_pairs True


## Runtime Summary

In [6]:
print('run | query_voxels | edges | voxelize_s | dense_s | sparse_s | dense_extra_mb | sparse_extra_mb | dense/sparse_mem | equivalent')
for row in results:
    print(
        row['run'],
        row['query_voxels'],
        row['edges'],
        f"{row['voxelize_seconds']:.4f}",
        f"{row['dense_seconds']:.4f}",
        f"{row['sparse_seconds']:.4f}",
        f"{row['dense_extra_peak_mb']:.1f}",
        f"{row['sparse_extra_peak_mb']:.1f}",
        f"{row['dense_extra_peak_mb'] / row['sparse_extra_peak_mb']:.3f}",
        row['same_pairs'],
    )

dense_times = torch.tensor([row['dense_seconds'] for row in results], dtype=torch.float64)
sparse_times = torch.tensor([row['sparse_seconds'] for row in results], dtype=torch.float64)
voxelize_times = torch.tensor([row['voxelize_seconds'] for row in results], dtype=torch.float64)
dense_extra_mem = torch.tensor([row['dense_extra_peak_mb'] for row in results], dtype=torch.float64)
sparse_extra_mem = torch.tensor([row['sparse_extra_peak_mb'] for row in results], dtype=torch.float64)
dense_peak_mem = torch.tensor([row['dense_peak_mb'] for row in results], dtype=torch.float64)
sparse_peak_mem = torch.tensor([row['sparse_peak_mb'] for row in results], dtype=torch.float64)

print('')
print('dense mean seconds =', dense_times.mean().item())
print('sparse mean seconds =', sparse_times.mean().item())
print('voxelize mean seconds =', voxelize_times.mean().item())
print('mean dense/sparse =', (dense_times / sparse_times).mean().item())
print('dense mean extra peak MB =', dense_extra_mem.mean().item())
print('sparse mean extra peak MB =', sparse_extra_mem.mean().item())
print('mean dense/sparse extra peak memory =', (dense_extra_mem / sparse_extra_mem).mean().item())
print('dense mean absolute peak MB =', dense_peak_mem.mean().item())
print('sparse mean absolute peak MB =', sparse_peak_mem.mean().item())
print('all runs equivalent =', all(row['same_pairs'] for row in results))

assert all(row['same_pairs'] for row in results)

run | query_voxels | edges | voxelize_s | dense_s | sparse_s | dense_extra_mb | sparse_extra_mb | dense/sparse_mem | equivalent
0 4619456 62848719 8.5022 0.2005 0.0421 10182.6 6666.0 1.528 True
1 4513573 61052038 8.6653 0.1778 0.0386 10133.6 6523.1 1.553 True
2 3773820 51990377 7.7324 0.1604 0.0356 9841.8 5529.5 1.780 True
3 4208605 57806222 7.6409 0.1607 0.0352 10026.6 6113.5 1.640 True
4 4420296 60903873 8.3229 0.1696 0.0395 10112.3 6397.5 1.581 True

dense mean seconds = 0.17379746437072754
sparse mean seconds = 0.038206624984741214
voxelize mean seconds = 8.172735166549682
mean dense/sparse = 4.545884795296435
dense mean extra peak MB = 10059.3966796875
sparse mean extra peak MB = 6245.906640625
mean dense/sparse extra peak memory = 1.6163368462076846
dense mean absolute peak MB = 10486.13779296875
sparse mean absolute peak MB = 6672.64775390625
all runs equivalent = True


## CPU Reference vs CUDA Backend

The CPU reference is single-threaded, so this check uses small random sparse lattices. Each run compares the bottom-level `kids_by_offset [Nq, L]` tensor from CPU and CUDA directly.

In [7]:
from symtrellis.geometry.neighbors import _sparse_lattice_ext as ext

assert hasattr(ext, 'radius_nbr_kids_by_offset_sparse_lattice_cpu')

cpu_gpu_rng = torch.Generator(device='cpu').manual_seed(seed + 2026)
cpu_gpu_results = []
cpu_gpu_radius = 2.0
cpu_gpu_offsets = lattice_ball_offsets(cpu_gpu_radius, device='cpu').contiguous()
cpu_gpu_n_runs = 5

for run_id in range(cpu_gpu_n_runs):
    local_grid_size = 32
    batch_count = 2
    keys_per_batch = 700
    queries_per_batch = 450
    coord_min_cpu = torch.randint(
        -512, 513, (3,), generator=cpu_gpu_rng, dtype=torch.int32
    )

    key_parts = []
    key_bid_parts = []
    query_parts = []
    query_bid_parts = []

    for bid in range(batch_count):
        lin = torch.randperm(local_grid_size ** 3, generator=cpu_gpu_rng)[:keys_per_batch]
        local_key = torch.stack(
            [
                lin % local_grid_size,
                (lin // local_grid_size) % local_grid_size,
                lin // (local_grid_size * local_grid_size),
            ],
            dim=1,
        ).to(torch.int32)
        key_parts.append(local_key + coord_min_cpu)
        key_bid_parts.append(torch.full((keys_per_batch,), bid, dtype=torch.int32))

        local_query = torch.rand(
            (queries_per_batch, 3), generator=cpu_gpu_rng, dtype=torch.float64
        ) * (local_grid_size - 1)
        query_parts.append(local_query + coord_min_cpu.to(torch.float64))
        query_bid_parts.append(torch.full((queries_per_batch,), bid, dtype=torch.int32))

    key_cpu = torch.cat(key_parts, dim=0).contiguous()
    key_bid_cpu = torch.cat(key_bid_parts, dim=0).contiguous()
    query_cpu = torch.cat(query_parts, dim=0).contiguous()
    query_bid_cpu = torch.cat(query_bid_parts, dim=0).contiguous()

    cpu_kids_by_offset = ext.radius_nbr_kids_by_offset_sparse_lattice_cpu(
        query_cpu,
        query_bid_cpu,
        key_cpu,
        key_bid_cpu,
        cpu_gpu_radius,
        cpu_gpu_offsets,
        coord_min_cpu,
    )
    cuda_kids_by_offset = ext.radius_nbr_kids_by_offset_sparse_lattice_cuda(
        query_cpu.to(device),
        query_bid_cpu.to(device),
        key_cpu.to(device),
        key_bid_cpu.to(device),
        cpu_gpu_radius,
        cpu_gpu_offsets.to(device),
        coord_min_cpu.to(device),
    ).cpu()
    torch.cuda.synchronize()

    same_shape = cpu_kids_by_offset.shape == cuda_kids_by_offset.shape
    same_dtype = cpu_kids_by_offset.dtype == cuda_kids_by_offset.dtype == torch.int64
    same_values = torch.equal(cpu_kids_by_offset, cuda_kids_by_offset)
    same = bool(same_shape and same_dtype and same_values)
    hits = int((cpu_kids_by_offset >= 0).sum().item())
    cpu_gpu_results.append({'run': run_id, 'same': same, 'hits': hits})

    print(
        'run', run_id,
        'shape', tuple(cpu_kids_by_offset.shape),
        'hits', hits,
        'same_kids_by_offset', same,
    )
    assert same

print('all CPU/CUDA backend runs equivalent =', all(row['same'] for row in cpu_gpu_results))
assert all(row['same'] for row in cpu_gpu_results)

run 0 shape (900, 88) hits 596 same_kids_by_offset True
run 1 shape (900, 88) hits 645 same_kids_by_offset True
run 2 shape (900, 88) hits 602 same_kids_by_offset True
run 3 shape (900, 88) hits 660 same_kids_by_offset True
run 4 shape (900, 88) hits 596 same_kids_by_offset True
all CPU/CUDA backend runs equivalent = True
